In [0]:
import importlib
import configs.constant as constants
importlib.reload(constants)

from configs.constant import * 
from pyspark.sql.functions import * 
# that event is injected from event hub using the connection string in the config file from microshoft azure portal
# Namespace = Apartment Building 
# Event Hub = Individual Rooms 


ehConf = {
  "kafka.bootstrap.servers": f"{NAMESPACE}.servicebus.windows.net:9093",
  "kafka.security.protocol": "SASL_SSL",
  "kafka.sasl.mechanism": "PLAIN",
  "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{CONNECTION_STRING}";',
  "subscribe": "retail-events",
  "startingOffsets": "earliest",
  "kafka.group.id": "databricks-consumer"
}

df = spark.readStream \
    .format("kafka") \
    .options(**ehConf) \
    .load()


In [0]:
# dbutils.secrets.list("kv-scope")
dbutils.secrets.listScopes()

In [0]:
display(df)

In [0]:
spark.conf.set(
  "fs.azure.account.key.adlsgen2detraining2026.dfs.core.windows.net",
  "hbxOwTCAgdRzpwMOhIFjtmxy6E6/J0bh9zD7FsGkrA62dNmvk9WPpL9VaFKIC5pDjUb4lvC5wECb+ASt2JdTww=="
)

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_bronze = df.selectExpr("CAST(value AS STRING)", "partition") \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("event_hub_partition", col("partition")) \
    .withColumn("source", lit("event_hub"))

In [0]:
display(df)

In [0]:
df_bronze.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"abfss://retail-bronze@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/checkpoints/kafka_bronze/") \
    .trigger(availableNow=True) \
    .start(f"abfss://retail-bronze@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/kafka_data/")